# T07: Retrieval-Strategien

Der Baseline-Retriever embeddet die Frage und liefert die nächstgelegenen Chunks im Embedding-Raum. Das funktioniert, wenn die Frage ähnlich formuliert ist wie die Dokumente. Es scheitert, wenn Fragen:

- auf exakten Kennungen oder Codes beruhen
- Bedeutung mit Stichwort-Bedingungen mischen
- auf viele Chunks vage, aber nur auf wenige genau passen
- ein bekanntes Dokument betreffen, aber alle durchsuchen

Das sichtbarste Symptom sind unterschiedliche Begriffe: Der richtige Chunk existiert, steht aber zu weit unten.

| Frage | Text im Dokument | Problem |
|---|---|---|
| `"FSC-C147829 certificate"` | `"reference: FSC-C147829"` | Embeddings verwischen exakte Codes |
| `"PFAS-free"` | `"no intentionally added per- and polyfluoroalkyl substances"` | Abkürzung vs. ausgeschriebener Name |

Dieses Notebook vergleicht drei Alternativen mit der Baseline:

| Strategie | Ansatz | Zusätzlicher Aufwand |
|---|---|---|
| Baseline | Semantische Suche | Keiner |
| BM25 | Stichwortsuche, ohne Embeddings | Index im Arbeitsspeicher |
| Hybrid | Semantisch + BM25 über Reciprocal Rank Fusion | Index im Arbeitsspeicher |
| Metadatenfilter | Semantische Suche innerhalb bekannter Dokumente | Keiner |

Query Rewriting folgt in T09. Code, Prompts und Dokumente sind auf Englisch (Begründung in T01).

## Setup

Benötigt `OPENAI_API_KEY` (lokal: in `.env.local` im Hauptverzeichnis des Repos). Zuerst T01 ausführen: Dort entsteht der Vector Store, den die späteren Notebooks wiederverwenden.

In [ ]:
import sys

from loguru import logger
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import AnswerRelevancy, Faithfulness
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings as LangChainOpenAIEmbeddings

from conversational_toolkit.evaluation import EvaluationSample
from conversational_toolkit.evaluation.adapters import evaluate_with_ragas
from ki_kompass_rag.advanced_retrieval import (
    compare_retrieval_strategies,
    print_strategy_comparison,
    rag_answer_from_chunks,
    retrieve_baseline,
    retrieve_with_metadata_filter,
)
from ki_kompass_rag.baseline_rag import (
    RETRIEVER_TOP_K,
    build_llm,
    load_or_build_vector_store,
)
from ki_kompass_rag.evaluation import EVALUATION_QUERIES

logger.remove()
logger.add(sys.stderr, level="WARNING")

vector_store, embedding_model = await load_or_build_vector_store()
llm = build_llm("openai")

## BM25

**BM25** sortiert Chunks nach exakt übereinstimmenden Tokens, gewichtet mit Statistiken über den Korpus. Embeddings kommen nicht vor. Es kombiniert:

- **Sättigung der Termhäufigkeit**: Mehr Vorkommen eines Begriffs helfen, aber immer weniger.
- **Längennormalisierung**: Lange Chunks werden nicht bevorzugt, nur weil sie mehr Wörter enthalten.
- **Inverse Dokumenthäufigkeit**: Seltene Begriffe (Codes, Abkürzungen, Produktnamen) zählen mehr als häufige Wörter.

Zwei Parameter steuern es: `k1` (Sättigung, typisch 1.2 bis 2.0) und `b` (Längennormalisierung, 0 bis 1). Der Index (Term- und Dokumenthäufigkeiten) wird einmal aus allen Chunks im Vector Store gebaut und im Arbeitsspeicher gehalten.

BM25 nimmt Begriffe wörtlich: "carbon footprint decrease" und "CO₂ reduction" haben kein Token gemeinsam, gelten also als unverwandt. Semantische Suche findet Umschreibungen, BM25 exakte Codes.

In [ ]:
QUERY = "FSC-C147829 certificate"
KEYWORDS = ["FSC", "C147829"]

results = await compare_retrieval_strategies(
    QUERY, embedding_model, vector_store, top_k=RETRIEVER_TOP_K
)
print_strategy_comparison(
    {name: results[name] for name in ("bm25", "baseline")},
    relevant_keywords=KEYWORDS,
    top_n=RETRIEVER_TOP_K,
)

✓ markiert Chunks, die eines der Stichwörter enthalten.

## Hybrides Retrieval: Semantisch + BM25

Hybrides Retrieval führt beide Retriever parallel aus und vereint ihre Ranglisten mit **Reciprocal Rank Fusion (RRF)**:

$$\text{RRF}(d) = \sum_{r \in \text{retrievers}} \frac{1}{k + \text{rank}_r(d)}, \quad k = 60$$

RRF verwendet Ränge, nicht Scores; Kosinus-Ähnlichkeiten und BM25-Scores müssen also nie auf eine Skala gebracht werden. Chunks, die beide Retriever finden, stehen zuoberst; Chunks von nur einem zählen trotzdem.

In [ ]:
QUERY = "Are any tape products free of per- and polyfluoroalkyl substances?"
KEYWORDS = ["PFAS", "polyfluoroalkyl"]

results = await compare_retrieval_strategies(
    QUERY, embedding_model, vector_store, top_k=RETRIEVER_TOP_K
)
print_strategy_comparison(results, relevant_keywords=KEYWORDS, top_n=RETRIEVER_TOP_K)

## Metadatenfilter

Die Strategien oben durchsuchen alle Dokumente. Betrifft die Frage ein bekanntes Dokument ("what are the transport emissions in the tesa EPD?"), bringen Chunks aus anderen Dokumenten nur Rauschen. Ein ChromaDB-`where`-Filter beschränkt die Suche vor dem Sortieren auf Chunks mit passenden Metadaten:

```python
{"source_file": {"$eq": "EPD_tape_IPG_hotmelt.pdf"}}                 # one file
{"$or": [{"source_file": {"$eq": "A.pdf"}},
         {"source_file": {"$eq": "B.pdf"}}]}                         # two files
```

Das passt, wenn die Nutzerin oder der Nutzer im UI ein Dokument ausgewählt hat oder ein Tool einen Dokumentnamen übergibt.

In [ ]:
records = await vector_store.get_chunks_by_filter()
print("\n".join(sorted({r.metadata["source_file"] for r in records})))

In [ ]:
TARGET_FILE = "EPD_cardboard_grupak_corrugated.pdf"
QUERY = "What are the environmental impacts of this product?"

filtered = await retrieve_with_metadata_filter(
    QUERY,
    embedding_model,
    vector_store,
    filters={"source_file": {"$eq": TARGET_FILE}},
    top_k=RETRIEVER_TOP_K,
)
unfiltered = await retrieve_baseline(
    QUERY, embedding_model, vector_store, top_k=RETRIEVER_TOP_K
)
print(f"Filtered to {TARGET_FILE}:\n{filtered}\n\nUnfiltered:\n{unfiltered}")

## Antworten pro Strategie

`rag_answer_from_chunks()` antwortet aus einer vorgegebenen Liste von Chunks mit dem System-Prompt aus T01; nur das Retrieval unterscheidet sich.

In [ ]:
QUERY = "What is the FSC-C147829 certificate?"
KEYWORDS = ["FSC", "C147829"]

results = await compare_retrieval_strategies(
    QUERY, embedding_model, vector_store, top_k=RETRIEVER_TOP_K
)
print_strategy_comparison(results, relevant_keywords=KEYWORDS, top_n=RETRIEVER_TOP_K)
for name, result in results.items():
    print(f"\n>>> {name}")
    print(await rag_answer_from_chunks(llm, result.chunks, QUERY))

## Antwortqualität pro Strategie

Für jede Frage des Testsets aus T04 sucht jede Strategie Chunks und erzeugt eine Antwort; RAGAS bewertet die Antworten dann mit den zwei Metriken ohne Referenzantwort aus T04 (Faithfulness, AnswerRelevancy). Das ergibt 21 Antworten plus die Bewertungsaufrufe, einige Minuten.

In [ ]:
ragas_llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-4o-mini", max_completion_tokens=8192)
)
ragas_embeddings = LangChainOpenAIEmbeddings(model="text-embedding-3-small")

STRATEGIES = ("baseline", "bm25", "hybrid")
samples: dict[str, list[EvaluationSample]] = {s: [] for s in STRATEGIES}
for query in (q["query"] for q in EVALUATION_QUERIES):
    results = await compare_retrieval_strategies(
        query, embedding_model, vector_store, top_k=RETRIEVER_TOP_K
    )
    for strategy in STRATEGIES:
        chunks = list(results[strategy].chunks)
        samples[strategy].append(
            EvaluationSample(
                query=query,
                answer=await rag_answer_from_chunks(llm, chunks, query),
                retrieved_chunks=chunks,
            )
        )

In [ ]:
metrics = [Faithfulness(), AnswerRelevancy(strictness=1)]
print(f"{'strategy':<10}" + "".join(f"{m.name:>20}" for m in metrics))
for strategy, strategy_samples in samples.items():
    report = evaluate_with_ragas(
        samples=strategy_samples,
        metrics=metrics,
        llm=ragas_llm,
        embeddings=ragas_embeddings,
    )
    print(f"{strategy:<10}" + "".join(f"{s:>20.3f}" for s in report.summary().values()))

## Zusammenfassung

| Strategie | Geeignet für | Grenze |
|---|---|---|
| Baseline (semantisch) | Umschreibungen, unterschiedliche Begriffe | Exakte Codes und IDs |
| BM25 | Exakte Begriffe, Produktcodes, Abkürzungen | Umschreibungen, Synonyme |
| Hybrid | Beides, zuverlässig | Index muss in den Arbeitsspeicher passen |
| Metadatenfilter | Fragen zu einem bekannten Dokument | Der Aufrufer muss das Dokument kennen |